# SDF-RT PYNQ Notebook

这个 notebook 按 `build/vivado/fpga_top.v` 当前实现控制 KV260 上的 `fpga_top`：

- 用 `sdf_global_mem.mem` / `sdf_local_mem.mem` 初始化 SDF
- 通过 setup 窗口写固定参数
- 通过 `frame_start` 脉冲启动一帧
- 配置 `axi_dma_0` 将像素流写到 PYNQ 分配的连续帧缓冲
- 从帧缓冲读回 `400x400` 图像并在 notebook 内直接显示


In [ ]:
import struct
import time
from array import array
from pathlib import Path

import numpy as np
from IPython.display import display
from PIL import Image

from pynq import MMIO, Overlay, allocate

MEM_DIR = Path("/home/ubuntu/code/SDF-RT/csrc/vivado_mem")
BITSTREAM_PATH = "/home/ubuntu/code/sdf_rt.xsa"
IP_NAME = "fpga_top_0"
DMA_NAME = "axi_dma_0"

FRAME_WIDTH = 400
FRAME_HEIGHT = 400
BYTES_PER_PIXEL = 4
FRAME_BUFFER_BYTES = FRAME_WIDTH * FRAME_HEIGHT * BYTES_PER_PIXEL
FRAME_BUFFER_WORDS = FRAME_BUFFER_BYTES // 4

# Fixed setup values.
SETUP_ORIGIN = (0.0, 0.4, 2.8)
SETUP_GRID_MIN = (-0.570754833, 0.0, -0.442573989)
SETUP_GRID_MAX = (0.5706041, 1.131161911, 0.4423496)

# Tight SDF address mapping from SdfMem.sv.
GLOBAL_SDF_WORDS = 4096
LOCAL_SDF_BASE_WORD = GLOBAL_SDF_WORDS
LOCAL_SDF_WORDS = 131072
LOCAL_CELL_COUNT = 2048
LOCAL_WORDS_PER_CELL = 64

# Setup/status window in fpga_top.v.
SETUP_BASE_OFFSET = 0xFFFC0
SETUP_REG0 = SETUP_BASE_OFFSET + 0x00
SETUP_REG1 = SETUP_BASE_OFFSET + 0x04
SETUP_REG2 = SETUP_BASE_OFFSET + 0x08
SETUP_REG3 = SETUP_BASE_OFFSET + 0x0C
SETUP_REG4 = SETUP_BASE_OFFSET + 0x10
SETUP_REG5 = SETUP_BASE_OFFSET + 0x14
SETUP_REG6 = SETUP_BASE_OFFSET + 0x18
SETUP_REG7 = SETUP_BASE_OFFSET + 0x1C
SETUP_REG8 = SETUP_BASE_OFFSET + 0x20
SETUP_REG9 = SETUP_BASE_OFFSET + 0x24
FRAME_CTRL_REG = SETUP_BASE_OFFSET + 0x30

# Status window in fpga_top.v.
STATUS_REG = SETUP_BASE_OFFSET + 0x28
FRAME_COUNT_REG = SETUP_BASE_OFFSET + 0x34

# AXI DMA S2MM registers (simple mode)
S2MM_DMACR = 0x30
S2MM_DMASR = 0x34
S2MM_DEST_ADDR = 0x48
S2MM_DEST_ADDR_MSB = 0x4C
S2MM_LENGTH = 0x58


In [ ]:
def float_to_u32(value):
    return struct.unpack(">I", struct.pack(">f", float(value)))[0]


def parse_mem_words(mem_path):
    words = []
    with open(mem_path, "r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith("//"):
                continue
            if line.startswith("@"):
                continue
            words.append(int(line, 16))
    return words


def frame_buffer_to_image(rgb_buffer, width, height):
    image = bytearray(width * height * 3)
    for pixel_idx, rgb in enumerate(rgb_buffer):
        base = pixel_idx * 3
        image[base + 0] = (rgb >> 16) & 0xFF
        image[base + 1] = (rgb >> 8) & 0xFF
        image[base + 2] = rgb & 0xFF
    return Image.frombytes("RGB", (width, height), bytes(image))


def display_frame(rgb_buffer, width, height):
    image = frame_buffer_to_image(rgb_buffer, width, height)
    display(image)
    return image


def decode_dma_status(status):
    flags = []
    if status & 0x00000001:
        flags.append("halted")
    if status & 0x00000002:
        flags.append("idle")
    if status & 0x00000010:
        flags.append("dma_internal_err")
    if status & 0x00000020:
        flags.append("dma_slave_err")
    if status & 0x00000040:
        flags.append("dma_decode_err")
    if status & 0x00000100:
        flags.append("sg_internal_err")
    if status & 0x00000200:
        flags.append("sg_slave_err")
    if status & 0x00000400:
        flags.append("sg_decode_err")
    if status & 0x00001000:
        flags.append("ioc_irq")
    if status & 0x00002000:
        flags.append("dly_irq")
    if status & 0x00004000:
        flags.append("err_irq")
    return ", ".join(flags) if flags else "none"


def decode_core_status(status):
    flags = []
    if status & 0x01:
        flags.append("setup_ready")
    if status & 0x02:
        flags.append("frame_done")
    if status & 0x04:
        flags.append("busy")
    if status & 0x08:
        flags.append("validation_error")
    if status & 0x10:
        flags.append("stall_detected")
    return ", ".join(flags) if flags else "none"


In [ ]:
class FpgaTopDriver:
    def __init__(
        self,
        bitstream_path=BITSTREAM_PATH,
        ip_name=IP_NAME,
        dma_name=DMA_NAME,
        frame_buffer_bytes=FRAME_BUFFER_BYTES,
    ):
        self.overlay = Overlay(bitstream_path)
        self.ip_name = ip_name
        self.dma_name = dma_name
        self.ip = getattr(self.overlay, ip_name, None)
        self.dma_ip = getattr(self.overlay, dma_name, None)

        if self.ip is None:
            raise KeyError(f"IP '{ip_name}' not found in overlay")
        if self.dma_ip is None:
            raise KeyError(f"IP '{dma_name}' not found in overlay")

        self.base_addr = self.overlay.ip_dict[ip_name]["phys_addr"]
        self.addr_range = self.overlay.ip_dict[ip_name]["addr_range"]
        self.dma_base_addr = self.overlay.ip_dict[dma_name]["phys_addr"]
        self.dma_addr_range = self.overlay.ip_dict[dma_name]["addr_range"]
        self.frame_buffer_bytes = frame_buffer_bytes
        self.frame_buffer = allocate(shape=(FRAME_BUFFER_WORDS,), dtype=np.uint32)
        self.frame_buffer_base_addr = self.frame_buffer.physical_address

        # Prefer overlay-managed MMIO objects so the notebook does not depend on raw /dev/mem access.
        self.mmio = getattr(self.ip, "mmio", None)
        if self.mmio is None:
            self.mmio = MMIO(self.base_addr, self.addr_range, device=self.overlay.device)

        self.dma_mmio = getattr(self.dma_ip, "mmio", None)
        if self.dma_mmio is None:
            self.dma_mmio = MMIO(self.dma_base_addr, self.dma_addr_range, device=self.overlay.device)

    def write(self, offset, value):
        self.mmio.write(offset, value & 0xFFFFFFFF)

    def read(self, offset):
        return self.mmio.read(offset) & 0xFFFFFFFF

    def write_word_addr(self, word_addr, value):
        self.write(word_addr << 2, value)

    def read_word_addr(self, word_addr):
        return self.read(word_addr << 2)

    def dma_write(self, offset, value):
        self.dma_mmio.write(offset, value & 0xFFFFFFFF)

    def dma_read(self, offset):
        return self.dma_mmio.read(offset) & 0xFFFFFFFF

    def read_status(self):
        return self.read(STATUS_REG) & 0xFFFFFFFF

    def read_frame_count(self):
        return self.read(FRAME_COUNT_REG) & 0xFFFFFFFF

    def print_info(self):
        print(f"overlay loaded   : {self.overlay.is_loaded()}")
        print(f"fpga_top name   : {self.ip_name}")
        print(f"fpga_top base   : 0x{self.base_addr:016X}")
        print(f"fpga_top range  : 0x{self.addr_range:016X}")
        print(f"setup_base      : 0x{self.base_addr + SETUP_BASE_OFFSET:016X}")
        print(f"dma name        : {self.dma_name}")
        print(f"dma base        : 0x{self.dma_base_addr:016X}")
        print(f"dma range       : 0x{self.dma_addr_range:016X}")
        print(f"frame buffer    : 0x{self.frame_buffer_base_addr:016X}")
        print(f"frame bytes     : {self.frame_buffer_bytes}")
        status = self.read_status()
        print(f"core status     : 0x{status:08X} ({decode_core_status(status)})")

    def smoke_test_setup_window(self):
        print("[AXI] sequential write/read smoke test on setup window")
        patterns = [
            (SETUP_REG1, 0x00000001),
            (SETUP_REG2, 0x00000002),
            (SETUP_REG3, 0x00000003),
            (SETUP_REG4, 0x00000004),
        ]
        for offset, value in patterns:
            self.write(offset, value)
        for offset, expected in patterns:
            actual = self.read(offset)
            if actual != expected:
                raise RuntimeError(
                    f"AXI smoke test failed at 0x{offset:08X}: expected 0x{expected:08X}, got 0x{actual:08X}"
                )
        print("[AXI] smoke test passed")

    def init_sdf(self, global_words, local_words, progress_step=4096):
        if len(global_words) != GLOBAL_SDF_WORDS:
            raise ValueError(
                f"global SDF size mismatch: expected {GLOBAL_SDF_WORDS}, got {len(global_words)}"
            )
        if len(local_words) > LOCAL_SDF_WORDS:
            raise ValueError(
                f"local SDF too large: max {LOCAL_SDF_WORDS}, got {len(local_words)}"
            )

        print(f"[SDF] writing global SDF: {len(global_words)} words")
        for idx, value in enumerate(global_words):
            self.write_word_addr(idx, value)
            if idx and idx % progress_step == 0:
                print(f"  global {idx}/{len(global_words)}")

        print(f"[SDF] writing local SDF: {len(local_words)} words")
        for idx, value in enumerate(local_words):
            self.write_word_addr(LOCAL_SDF_BASE_WORD + idx, value)
            if idx and idx % progress_step == 0:
                print(f"  local {idx}/{len(local_words)}")

        print("[SDF] initialization complete")

    def program_setup(self, origin=SETUP_ORIGIN, grid_min=SETUP_GRID_MIN, grid_max=SETUP_GRID_MAX):
        setup_values = [
            float_to_u32(origin[0]),
            float_to_u32(origin[1]),
            float_to_u32(origin[2]),
            float_to_u32(grid_min[0]),
            float_to_u32(grid_min[1]),
            float_to_u32(grid_min[2]),
            float_to_u32(grid_max[0]),
            float_to_u32(grid_max[1]),
            float_to_u32(grid_max[2]),
        ]
        setup_regs = [
            SETUP_REG1,
            SETUP_REG2,
            SETUP_REG3,
            SETUP_REG4,
            SETUP_REG5,
            SETUP_REG6,
            SETUP_REG7,
            SETUP_REG8,
            SETUP_REG9,
        ]

        print("[SETUP] writing fixed setup values")
        for reg, value in zip(setup_regs, setup_values):
            self.write(reg, value)
            rb = self.read(reg)
            if rb != value:
                raise RuntimeError(
                    f"setup write verify failed at 0x{reg:08X}: expected 0x{value:08X}, got 0x{rb:08X}"
                )

        self.write(SETUP_REG0, 0x1)
        self.write(SETUP_REG0, 0x0)
        print("[SETUP] setup_valid pulse sent")

    def wait_setup_ready(self, timeout_s=1.0, poll_interval=0.001):
        start = time.time()
        while True:
            status = self.read_status()
            if status & 0x01:
                print(f"[SETUP] ready: status=0x{status:08X} ({decode_core_status(status)})")
                return status
            if time.time() - start > timeout_s:
                raise TimeoutError(
                    f"Setup timeout: status=0x{status:08X} ({decode_core_status(status)})"
                )
            time.sleep(poll_interval)

    def start_frame(self):
        self.write(FRAME_CTRL_REG, 0x1)
        self.write(FRAME_CTRL_REG, 0x0)
        print("[FRAME] frame_start pulse sent")

    def clear_frame_buffer(self):
        self.frame_buffer[:] = 0
        self.frame_buffer.flush()

    def start_s2mm_transfer(self, dest_addr=None, length=None):
        dest_addr = self.frame_buffer_base_addr if dest_addr is None else dest_addr
        length = self.frame_buffer_bytes if length is None else length

        dest_addr_hi = (dest_addr >> 32) & 0xFFFFFFFF
        dest_addr_lo = dest_addr & 0xFFFFFFFF

        self.dma_write(S2MM_DMACR, 0x4)
        time.sleep(0.001)
        self.dma_write(S2MM_DMASR, 0x00007000)
        self.dma_write(S2MM_DMACR, 0x00000001)
        self.dma_write(S2MM_DEST_ADDR, dest_addr_lo)
        self.dma_write(S2MM_DEST_ADDR_MSB, dest_addr_hi)
        self.dma_write(S2MM_LENGTH, length)

        dmacr = self.dma_read(S2MM_DMACR)
        status = self.dma_read(S2MM_DMASR)
        print(f"[DMA] dest=0x{dest_addr:016X} len={length}")
        print(f"[DMA] dmacr=0x{dmacr:08X}")
        print(f"[DMA] S2MM start: status=0x{status:08X} ({decode_dma_status(status)})")

    def wait_s2mm_done(self, timeout_s=5.0, poll_interval=0.001):
        start = time.time()
        while True:
            status = self.dma_read(S2MM_DMASR)
            core_status = self.read_status()
            if status & 0x00004000:
                raise RuntimeError(
                    f"DMA error: status=0x{status:08X} ({decode_dma_status(status)})"
                )
            if (status & 0x00001000) or (status & 0x00000002):
                self.dma_write(S2MM_DMASR, 0x00007000)
                print(f"[DMA] S2MM done: status=0x{status:08X} ({decode_dma_status(status)})")
                print(f"[CORE] status=0x{core_status:08X} ({decode_core_status(core_status)})")
                return status
            if time.time() - start > timeout_s:
                raise TimeoutError(
                    f"DMA timeout: dma=0x{status:08X} ({decode_dma_status(status)}), core=0x{core_status:08X} ({decode_core_status(core_status)})"
                )
            time.sleep(poll_interval)

    def read_frame_buffer(self, width=FRAME_WIDTH, height=FRAME_HEIGHT):
        total_pixels = width * height
        self.frame_buffer.invalidate()
        print(f"[READ] reading {total_pixels} pixels from allocated frame buffer")
        return array("I", (int(pixel) & 0x00FFFFFF for pixel in self.frame_buffer[:total_pixels]))


In [ ]:
global_mem_path = MEM_DIR / "sdf_global_mem.mem"
local_mem_path = MEM_DIR / "sdf_local_mem.mem"

global_words = parse_mem_words(global_mem_path)
local_words = parse_mem_words(local_mem_path)

print(f"[MEM] global words          = {len(global_words)}")
print(f"[MEM] local words           = {len(local_words)}")
print(f"[MEM] global capacity words = {GLOBAL_SDF_WORDS}")
print(f"[MEM] local capacity words  = {LOCAL_SDF_WORDS}")
print(f"[MEM] local cell count      = {LOCAL_CELL_COUNT}")
print(f"[MEM] words per local cell  = {LOCAL_WORDS_PER_CELL}")


In [ ]:
driver = FpgaTopDriver()
driver.print_info()


In [ ]:
driver.smoke_test_setup_window()


In [ ]:
start = time.time()
driver.init_sdf(global_words, local_words)
print(f"[SDF] elapsed {time.time() - start:.3f}s")


In [ ]:
driver.program_setup()
driver.wait_setup_ready(timeout_s=1.0)
driver.clear_frame_buffer()
driver.start_s2mm_transfer()
driver.start_frame()
driver.wait_s2mm_done(timeout_s=10.0)


In [ ]:
pixel_buffer = driver.read_frame_buffer()
frame_image = display_frame(pixel_buffer, FRAME_WIDTH, FRAME_HEIGHT)
frame_image
